# Proximal Policy Optimization (PPO): A Deep Dive

**Welcome to the PPO Deep Dive!**

In this notebook, we'll explore one of the most successful and widely-used algorithms in modern reinforcement learning. PPO has become the de facto standard for many RL applications because it's:
- **Effective**: Achieves strong performance across many tasks
- **Stable**: More robust than many other policy gradient methods
- **Simple**: Relatively straightforward to implement and tune
- **Sample efficient**: Makes good use of collected experience

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the motivation and intuition behind PPO
2. Master the mathematical foundations of PPO's key components
3. Implement PPO from scratch in PyTorch
4. Connect every line of code to the underlying theory
5. Train and analyze a PPO agent on a real environment

Let's begin!

---
# Part 1: Motivation and High-Level Intuition

## 1.1 The Challenge of Policy Optimization

In reinforcement learning, we want to find a policy $\pi_\theta(a|s)$ that maximizes expected cumulative reward:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^T \gamma^t r_t \right]$$

The **policy gradient** approach uses gradient ascent:

$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

### The REINFORCE Algorithm

The basic policy gradient (REINFORCE) uses:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]$$

where $G_t = \sum_{t'=t}^T \gamma^{t'-t} r_{t'}$ is the return from time $t$.

### Problems with Vanilla Policy Gradient

While simple and effective, vanilla policy gradient suffers from several issues:

**1. High Variance** 📊
- Returns $G_t$ can vary wildly between trajectories
- This leads to noisy gradient estimates
- Learning becomes slow and unstable

**2. Sample Inefficiency** 🗑️
- Each trajectory is used only once for a single gradient update
- We throw away expensive experience data after one use
- Requires many environment interactions

**3. Destructive Updates** 💥
- Large policy updates can drastically change behavior
- One bad update can ruin a previously good policy
- Policy can "collapse" to poor local optima

**4. Sensitive to Learning Rate** ⚖️
- Too large: unstable, divergent behavior
- Too small: extremely slow learning
- No principled way to choose step size

## 1.2 Trust Region Methods: A Better Way

The key insight: **Don't let the policy change too much in one update**.

### Trust Region Policy Optimization (TRPO)

TRPO solves an optimization problem:

$$\max_\theta \; \mathbb{E}_{s,a \sim \pi_{\theta_{\text{old}}}} \left[ \frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} A^{\pi_{\theta_{\text{old}}}}(s,a) \right]$$

subject to: $\mathbb{E}_s \left[ D_{\text{KL}}(\pi_{\theta_{\text{old}}}(\cdot|s) \| \pi_\theta(\cdot|s)) \right] \leq \delta$

**Translation**: Maximize expected advantage, but keep the new policy close to the old one (measured by KL divergence).

**Problem with TRPO**:
- Complex to implement (conjugate gradients, line search)
- Computationally expensive
- Hard to use with large neural networks

## 1.3 PPO: The Best of Both Worlds

PPO achieves TRPO's stability with REINFORCE's simplicity!

### Key Innovation: Clipped Surrogate Objective

Instead of a hard constraint, PPO uses a **clipped objective function**:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]$$

where:
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ is the probability ratio
- $\epsilon$ is a hyperparameter (typically 0.1 or 0.2)
- $A_t$ is the advantage estimate

**Why this works**:
1. When advantage is positive (good action): clip prevents excessive probability increase
2. When advantage is negative (bad action): clip prevents excessive probability decrease
3. Simple to implement: just a min and clip operation!
4. No expensive KL divergence computation needed

### The Three Pillars of PPO

1. **Clipped Surrogate Objective**: Prevents destructive policy updates
2. **Value Function Learning**: Reduces variance through learned baselines
3. **Generalized Advantage Estimation (GAE)**: Balances bias and variance

Let's dive into each component in detail!

---
# Part 2: Mathematical Foundations

## 2.1 The Probability Ratio: $r_t(\theta)$

The foundation of PPO is the **importance sampling ratio**:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

### What does this ratio tell us?

- $r_t(\theta) > 1$: New policy assigns **higher** probability to action $a_t$ than old policy
- $r_t(\theta) = 1$: New policy assigns **same** probability (no change)
- $r_t(\theta) < 1$: New policy assigns **lower** probability to action $a_t$

### Why use the ratio?

The ratio allows us to **reuse old trajectories** collected under $\pi_{\theta_{\text{old}}}$ to estimate gradients for $\pi_\theta$.

**Importance Sampling Identity**:

$$\mathbb{E}_{a \sim \pi_\theta}[f(a)] = \mathbb{E}_{a \sim \pi_{\theta_{\text{old}}}} \left[ \frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} f(a) \right]$$

This lets us use samples from the old policy to evaluate the new policy!

## 2.2 The Clipped Surrogate Objective

### Unclipped Surrogate Objective

First, consider the basic surrogate objective from TRPO:

$$L^{\text{CPI}}(\theta) = \mathbb{E}_t \left[ r_t(\theta) A_t \right] = \mathbb{E}_t \left[ \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)} A_t \right]$$

**CPI** stands for "Conservative Policy Iteration".

**Problem**: Without constraints, this objective can lead to excessively large policy updates.

### PPO's Clipped Objective

PPO introduces a clipped version:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]$$

where the clip function is:

$$\text{clip}(r, 1-\epsilon, 1+\epsilon) = \begin{cases}
1-\epsilon & \text{if } r < 1-\epsilon \\
r & \text{if } 1-\epsilon \leq r \leq 1+\epsilon \\
1+\epsilon & \text{if } r > 1+\epsilon
\end{cases}$$

### Understanding the Clipping: Case Analysis

Let's analyze what happens in different cases:

#### Case 1: $A_t > 0$ (Good Action)

We want to increase $\pi_\theta(a_t|s_t)$, i.e., make $r_t(\theta) > 1$.

- **Unclipped term**: $r_t(\theta) A_t$ keeps growing as $r_t$ increases
- **Clipped term**: $(1+\epsilon) A_t$ stops growing when $r_t > 1+\epsilon$
- **Minimum**: Takes the clipped term when $r_t > 1+\epsilon$

**Effect**: Prevents the new policy from making the action much more likely than the old policy.

#### Case 2: $A_t < 0$ (Bad Action)

We want to decrease $\pi_\theta(a_t|s_t)$, i.e., make $r_t(\theta) < 1$.

- **Unclipped term**: $r_t(\theta) A_t$ becomes more negative as $r_t$ decreases
- **Clipped term**: $(1-\epsilon) A_t$ stops becoming more negative when $r_t < 1-\epsilon$
- **Minimum**: Takes the clipped term when $r_t < 1-\epsilon$

**Effect**: Prevents the new policy from making the action much less likely than the old policy.

### Geometric Interpretation

The objective creates a "trust region" around the old policy:
- Within $[1-\epsilon, 1+\epsilon]$: normal policy gradient
- Outside this range: gradient becomes zero (clipping activates)

This is a **soft constraint** that's easier to optimize than TRPO's hard KL constraint!

## 2.3 Value Function and Critic Learning

To reduce variance, PPO uses a **value function** $V^\pi(s)$ as a baseline.

### TD Target

The temporal difference target is:

$$\hat{V}_t = r_t + \gamma V(s_{t+1})$$

For terminal states: $\hat{V}_t = r_t$ (since $V(s_{\text{terminal}}) = 0$).

### Value Function Loss

We train the value function to minimize:

$$L^{V}(\theta) = \mathbb{E}_t \left[ (V_\theta(s_t) - \hat{V}_t)^2 \right]$$

In practice, we often use **Huber loss** (smooth L1) for better stability:

$$L^{V}(\theta) = \mathbb{E}_t \left[ \text{SmoothL1}(V_\theta(s_t) - \hat{V}_t) \right]$$

## 2.4 Generalized Advantage Estimation (GAE)

### Why Advantage Functions?

The advantage function measures how much better an action is compared to the average:

$$A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$$

Using advantages instead of returns:
- Reduces variance (we subtract the state value baseline)
- Centers the gradient estimates around zero
- Makes learning more stable

### TD Error

The one-step TD error is:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is an **unbiased but high-variance** estimate of the advantage.

### GAE Formula

GAE exponentially weights TD errors:

$$A_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

Expanding this:

$$A_t^{\text{GAE}} = \delta_t + (\gamma \lambda) \delta_{t+1} + (\gamma \lambda)^2 \delta_{t+2} + \ldots$$

### Recursive Computation

We can compute GAE recursively (backwards through time):

$$A_t^{\text{GAE}} = \delta_t + (\gamma \lambda) A_{t+1}^{\text{GAE}}$$

With boundary condition: $A_T^{\text{GAE}} = \delta_T$ at the last timestep.

**Algorithm**:
```
1. Initialize: advantage = 0
2. For t = T-1 down to 0:
       advantage = δ_t + γλ * advantage
       A_t = advantage
```

### Hyperparameter $\lambda$

- $\lambda = 0$: $A_t = \delta_t$ (high bias, low variance)
- $\lambda = 1$: $A_t = \sum_l \gamma^l \delta_{t+l} = G_t - V(s_t)$ (low bias, high variance)
- $\lambda \in (0,1)$: trades off bias and variance

Typical value: $\lambda = 0.95$

## 2.5 Complete PPO Loss Function

The total loss combines three terms:

$$L(\theta) = \mathbb{E}_t \left[ L^{\text{CLIP}}(\theta) - c_1 L^V(\theta) + c_2 S[\pi_\theta](s_t) \right]$$

where:
- $L^{\text{CLIP}}(\theta)$: Clipped policy objective (we maximize this)
- $L^V(\theta)$: Value function loss (we minimize this, hence the minus)
- $S[\pi_\theta](s_t)$: Entropy bonus (encourages exploration)
- $c_1, c_2$: Coefficients (typically $c_1 = 0.5$, $c_2 = 0.01$)

In practice, we often:
1. Maximize: $-L^{\text{CLIP}}(\theta)$ (negate and minimize)
2. Minimize: $L^V(\theta)$
3. Maximize: Entropy (or minimize negative entropy)

## 2.6 PPO Algorithm Pseudocode

```
Initialize policy network π_θ and value network V_θ (or shared network)

for iteration = 1, 2, 3, ... do:
    # Step 1: Collect Trajectories
    Run policy π_θ_old in environment for T timesteps
    Store (s_t, a_t, r_t, s_t+1, π_θ_old(a_t|s_t))
    
    # Step 2: Compute Advantages
    Compute TD targets: V̂_t = r_t + γV(s_t+1)
    Compute TD errors: δ_t = r_t + γV(s_t+1) - V(s_t)
    Compute GAE advantages: A_t using λ and δ_t
    Normalize advantages: A_t = (A_t - mean(A)) / (std(A) + ε)
    
    # Step 3: Update Policy and Value Function
    for epoch = 1 to K do:
        for mini-batch in data do:
            # Compute ratio
            r_t = π_θ(a_t|s_t) / π_θ_old(a_t|s_t)
            
            # Compute losses
            L_CLIP = min(r_t * A_t, clip(r_t, 1-ε, 1+ε) * A_t)
            L_V = (V_θ(s_t) - V̂_t)²
            L_total = -L_CLIP + c₁ * L_V - c₂ * entropy
            
            # Update parameters
            θ ← θ - α∇_θ L_total
        
    # Step 4: Update old policy
    θ_old ← θ
```

Key aspects:
- **Multiple epochs** (K=3-10): Reuse trajectories multiple times
- **Mini-batches**: Process data in small chunks for better gradient estimates
- **Advantage normalization**: Critical for stable training
- **Clipping**: Prevents destructive updates

---
# Part 3: Implementation from Scratch

Now let's implement PPO in PyTorch! We'll build it step by step, explaining each component.

In [ ]:
# Import necessary libraries
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from collections import deque

# Set random seeds for reproducibility
np.random.seed(0)
torch.manual_seed(0)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

## 3.1 Hyperparameters

Let's define our hyperparameters with typical PPO values:

In [ ]:
# ============================================
# Hyperparameters
# ============================================

# Training
learning_rate = 3e-4       # Learning rate for Adam optimizer
gamma = 0.99               # Discount factor for future rewards
lmbda = 0.95               # GAE lambda parameter (bias-variance tradeoff)
eps_clip = 0.2             # Clipping parameter for PPO objective
K_epochs = 10              # Number of epochs to update policy
batch_size = 64            # Mini-batch size for training

# Loss coefficients
c1 = 0.5                   # Value function loss coefficient
c2 = 0.01                  # Entropy bonus coefficient

# Trajectory collection
T_horizon = 2048           # Timesteps per trajectory collection

# Environment
env_name = 'CartPole-v1'

print("Hyperparameters set:")
print(f"  Learning rate: {learning_rate}")
print(f"  Discount (γ): {gamma}")
print(f"  GAE λ: {lmbda}")
print(f"  Clipping ε: {eps_clip}")
print(f"  Epochs: {K_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Trajectory horizon: {T_horizon}")

## 3.2 Neural Network Architecture

We'll use a **shared network** architecture where:
- One shared feature extractor (hidden layers)
- Two heads: one for policy $\pi_\theta(a|s)$, one for value $V_\theta(s)$

**Why shared?**
- More parameter efficient
- Features learned for value prediction help policy
- Faster training

**Alternative**: Separate networks (more stable but slower)

In [ ]:
class PPO(nn.Module):
    """
    PPO Agent with shared network architecture.
    
    Architecture:
        Input (state) → FC1 → ReLU → FC2 → ReLU → { Policy Head, Value Head }
    """
    def __init__(self, state_dim=4, action_dim=2, hidden_dim=256):
        super(PPO, self).__init__()
        
        # Store dimensions
        self.state_dim = state_dim
        self.action_dim = action_dim
        
        # Shared feature extractor
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        
        # Policy head (actor): outputs action probabilities
        self.fc_pi = nn.Linear(hidden_dim, action_dim)
        
        # Value head (critic): outputs state value
        self.fc_v = nn.Linear(hidden_dim, 1)
        
        # Optimizer
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        
        # Trajectory buffer
        self.data = []
        
    def forward_features(self, x):
        """Extract features from input state."""
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x
    
    def pi(self, x, softmax_dim=0):
        """
        Policy network (actor).
        
        Args:
            x: State tensor
            softmax_dim: Dimension to apply softmax (0 for single state, 1 for batch)
        
        Returns:
            Action probability distribution
        """
        features = self.forward_features(x)
        logits = self.fc_pi(features)
        probs = F.softmax(logits, dim=softmax_dim)
        return probs
    
    def v(self, x):
        """
        Value network (critic).
        
        Args:
            x: State tensor
        
        Returns:
            State value estimate V(s)
        """
        features = self.forward_features(x)
        value = self.fc_v(features)
        return value
    
    def put_data(self, transition):
        """Store a transition in the buffer."""
        self.data.append(transition)
        
    def make_batch(self):
        """
        Convert stored transitions into training batches.
        
        Returns:
            Tensors of states, actions, rewards, next_states, done_masks, old_probs
        """
        s_lst, a_lst, r_lst, s_prime_lst, prob_a_lst, done_lst = [], [], [], [], [], []
        
        for transition in self.data:
            s, a, r, s_prime, prob_a, done = transition
            
            s_lst.append(s)
            a_lst.append([a])
            r_lst.append([r])
            s_prime_lst.append(s_prime)
            prob_a_lst.append([prob_a])
            
            # done_mask: 0 if episode ended, 1 otherwise (for TD target calculation)
            done_mask = 0.0 if done else 1.0
            done_lst.append([done_mask])
        
        # Convert to tensors
        s = torch.tensor(s_lst, dtype=torch.float)
        a = torch.tensor(a_lst)
        r = torch.tensor(r_lst, dtype=torch.float)
        s_prime = torch.tensor(s_prime_lst, dtype=torch.float)
        done_mask = torch.tensor(done_lst, dtype=torch.float)
        prob_a = torch.tensor(prob_a_lst, dtype=torch.float)
        
        # Clear buffer
        self.data = []
        
        return s, a, r, s_prime, done_mask, prob_a

# Create model instance
model = PPO(state_dim=4, action_dim=2, hidden_dim=256)
print("\nPPO Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3.3 Computing Advantages with GAE

This is a crucial function that implements the GAE computation we derived earlier.

**Recall the algorithm**:
1. Compute TD errors: $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$
2. Compute advantages backwards: $A_t = \delta_t + \gamma \lambda A_{t+1}$

In [ ]:
def compute_gae(rewards, values, next_values, dones, gamma=0.99, lmbda=0.95):
    """
    Compute Generalized Advantage Estimation.
    
    Args:
        rewards: Tensor of rewards [T, 1]
        values: Tensor of value estimates V(s_t) [T, 1]
        next_values: Tensor of next value estimates V(s_{t+1}) [T, 1]
        dones: Tensor of done masks [T, 1] (0 if done, 1 otherwise)
        gamma: Discount factor
        lmbda: GAE lambda parameter
    
    Returns:
        advantages: Tensor of advantage estimates [T, 1]
        returns: Tensor of TD targets (for value function training) [T, 1]
    """
    T = rewards.shape[0]
    advantages = torch.zeros_like(rewards)
    
    # Step 1: Compute TD errors
    # δ_t = r_t + γ * V(s_{t+1}) * done_mask - V(s_t)
    td_errors = rewards + gamma * next_values * dones - values
    
    # Step 2: Compute advantages using GAE (backwards through time)
    advantage = 0.0
    for t in reversed(range(T)):
        # A_t = δ_t + γλ * done_mask * A_{t+1}
        advantage = td_errors[t] + gamma * lmbda * dones[t] * advantage
        advantages[t] = advantage
    
    # TD targets for value function: V̂_t = A_t + V(s_t)
    # This is equivalent to: V̂_t = r_t + γV(s_{t+1}) + (A_t - δ_t)
    # But simpler to compute as: V̂_t = A_t + V(s_t)
    returns = advantages + values
    
    return advantages, returns

# Test the function with dummy data
dummy_rewards = torch.tensor([[1.0], [1.0], [1.0], [0.0]])
dummy_values = torch.tensor([[0.5], [0.6], [0.7], [0.0]])
dummy_next_values = torch.tensor([[0.6], [0.7], [0.0], [0.0]])
dummy_dones = torch.tensor([[1.0], [1.0], [0.0], [0.0]])

adv, ret = compute_gae(dummy_rewards, dummy_values, dummy_next_values, dummy_dones)
print("GAE computation test:")
print(f"Advantages shape: {adv.shape}")
print(f"Returns shape: {ret.shape}")
print("Test passed!")

## 3.4 The Training Function

This is where all the PPO magic happens! Let's implement the training loop with:
1. Computing advantages and returns
2. Mini-batch training
3. Clipped objective
4. Multiple epochs

In [ ]:
def train_ppo(model, K_epochs=10, batch_size=64, eps_clip=0.2, c1=0.5, c2=0.01):
    """
    Train PPO agent using collected trajectory data.
    
    Args:
        model: PPO model instance
        K_epochs: Number of epochs to train on the data
        batch_size: Mini-batch size
        eps_clip: Clipping parameter ε
        c1: Value function loss coefficient
        c2: Entropy bonus coefficient
    """
    # ============================================
    # Step 1: Prepare data
    # ============================================
    s, a, r, s_prime, done_mask, old_prob_a = model.make_batch()
    
    # ============================================
    # Step 2: Compute advantages using GAE
    # ============================================
    with torch.no_grad():
        # Get value estimates
        values = model.v(s)
        next_values = model.v(s_prime)
        
        # Compute GAE advantages and returns
        advantages, returns = compute_gae(r, values, next_values, done_mask, 
                                          gamma=gamma, lmbda=lmbda)
        
        # Normalize advantages (IMPORTANT for stable training!)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    # ============================================
    # Step 3: Train for K epochs
    # ============================================
    dataset_size = s.shape[0]
    
    for epoch in range(K_epochs):
        # Shuffle indices for mini-batch sampling
        indices = torch.randperm(dataset_size)
        
        # Train on mini-batches
        for start_idx in range(0, dataset_size, batch_size):
            # Get mini-batch indices
            end_idx = min(start_idx + batch_size, dataset_size)
            batch_indices = indices[start_idx:end_idx]
            
            # Extract mini-batch
            s_batch = s[batch_indices]
            a_batch = a[batch_indices]
            old_prob_a_batch = old_prob_a[batch_indices]
            advantages_batch = advantages[batch_indices]
            returns_batch = returns[batch_indices]
            
            # ============================================
            # Step 4: Compute policy and value predictions
            # ============================================
            
            # Get current policy probabilities
            pi = model.pi(s_batch, softmax_dim=1)  # [batch_size, action_dim]
            
            # Get probabilities of actions that were taken
            pi_a = pi.gather(1, a_batch)  # [batch_size, 1]
            
            # Get current value estimates
            values_batch = model.v(s_batch)  # [batch_size, 1]
            
            # ============================================
            # Step 5: Compute probability ratio
            # ============================================
            # r_t(θ) = π_θ(a_t|s_t) / π_θ_old(a_t|s_t)
            # We compute in log space for numerical stability:
            # r_t = exp(log(π_θ) - log(π_θ_old))
            ratio = torch.exp(torch.log(pi_a) - torch.log(old_prob_a_batch))
            
            # ============================================
            # Step 6: Compute clipped surrogate objective
            # ============================================
            # Unclipped objective: r_t * A_t
            surr1 = ratio * advantages_batch
            
            # Clipped objective: clip(r_t, 1-ε, 1+ε) * A_t
            surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantages_batch
            
            # Take minimum (pessimistic bound)
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # ============================================
            # Step 7: Compute value function loss
            # ============================================
            # Using smooth L1 loss for better stability
            value_loss = F.smooth_l1_loss(values_batch, returns_batch)
            
            # ============================================
            # Step 8: Compute entropy bonus
            # ============================================
            # Entropy encourages exploration
            # H(π) = -Σ π(a|s) log π(a|s)
            entropy = -(pi * torch.log(pi + 1e-8)).sum(dim=1).mean()
            
            # ============================================
            # Step 9: Total loss
            # ============================================
            # Loss = -L_CLIP + c1 * L_V - c2 * Entropy
            # (We negate policy loss because we want to maximize it)
            total_loss = policy_loss + c1 * value_loss - c2 * entropy
            
            # ============================================
            # Step 10: Gradient descent
            # ============================================
            model.optimizer.zero_grad()
            total_loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            
            model.optimizer.step()

print("Training function defined successfully!")

## 3.5 Complete Training Loop

Now let's put everything together and train an agent!

In [ ]:
def train_agent(n_episodes=500, T_horizon=2048, print_interval=10):
    """
    Main training loop for PPO agent.
    
    Args:
        n_episodes: Number of episodes to train
        T_horizon: Number of timesteps to collect before training
        print_interval: How often to print statistics
    
    Returns:
        episode_rewards: List of episode rewards
    """
    # Create environment and model
    env = gym.make(env_name)
    model = PPO(state_dim=4, action_dim=2, hidden_dim=256)
    
    # Tracking
    episode_rewards = []
    episode_reward = 0.0
    episode_count = 0
    recent_rewards = deque(maxlen=100)
    
    # Training loop
    timestep = 0
    state, _ = env.reset()
    
    print("Starting training...\n")
    
    while episode_count < n_episodes:
        # ============================================
        # Step 1: Collect trajectory
        # ============================================
        for t in range(T_horizon):
            timestep += 1
            
            # Select action using current policy
            with torch.no_grad():
                state_tensor = torch.from_numpy(state).float()
                prob = model.pi(state_tensor)
                m = Categorical(prob)
                action = m.sample().item()
                action_prob = prob[action].item()
            
            # Take action in environment
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition
            # Note: Normalize reward by dividing by 100 for CartPole
            model.put_data((state, action, reward/100.0, next_state, action_prob, done))
            
            # Update state and tracking
            state = next_state
            episode_reward += reward
            
            # Handle episode end
            if done:
                episode_rewards.append(episode_reward)
                recent_rewards.append(episode_reward)
                episode_count += 1
                
                # Print progress
                if episode_count % print_interval == 0:
                    avg_reward = np.mean(recent_rewards)
                    print(f"Episode {episode_count:4d} | "
                          f"Timestep {timestep:6d} | "
                          f"Avg Reward (last {len(recent_rewards)}): {avg_reward:.1f}")
                
                # Reset episode
                episode_reward = 0.0
                state, _ = env.reset()
                
                # Stop if we've reached target episodes
                if episode_count >= n_episodes:
                    break
        
        # ============================================
        # Step 2: Train on collected data
        # ============================================
        if len(model.data) > 0:  # Only train if we have data
            train_ppo(model, K_epochs=K_epochs, batch_size=batch_size, 
                     eps_clip=eps_clip, c1=c1, c2=c2)
    
    env.close()
    print("\nTraining complete!")
    
    return episode_rewards, model

print("Main training loop defined!")
print("Ready to train. Run the next cell to start training.")

## 3.6 Train the Agent!

Now let's actually train our PPO agent. This will take a few minutes.

**What to expect:**
- CartPole-v1 is solved when the average reward reaches 475+
- Our agent should solve it within 200-300 episodes
- You'll see the average reward steadily increase

In [ ]:
# Train the agent
print("="*60)
print("TRAINING PPO AGENT ON CARTPOLE-V1")
print("="*60)

episode_rewards, trained_model = train_agent(
    n_episodes=300,
    T_horizon=T_horizon,
    print_interval=10
)

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

## 3.7 Visualize Results

Let's plot the learning curve to see how our agent improved over time.

In [ ]:
def plot_learning_curve(rewards, window=10):
    """
    Plot the learning curve with moving average.
    
    Args:
        rewards: List of episode rewards
        window: Window size for moving average
    """
    plt.figure(figsize=(12, 6))
    
    # Raw rewards
    plt.plot(rewards, alpha=0.3, label='Episode Reward')
    
    # Moving average
    moving_avg = []
    for i in range(len(rewards)):
        start_idx = max(0, i - window + 1)
        moving_avg.append(np.mean(rewards[start_idx:i+1]))
    
    plt.plot(moving_avg, linewidth=2, label=f'Moving Average (window={window})')
    
    # Solved threshold for CartPole-v1
    plt.axhline(y=475, color='r', linestyle='--', linewidth=2, label='Solved Threshold (475)')
    
    plt.xlabel('Episode', fontsize=12)
    plt.ylabel('Reward', fontsize=12)
    plt.title('PPO Training on CartPole-v1', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\nTraining Statistics:")
    print(f"  Final 10-episode average: {np.mean(rewards[-10:]):.1f}")
    print(f"  Final 100-episode average: {np.mean(rewards[-100:]):.1f}")
    print(f"  Maximum reward achieved: {max(rewards):.1f}")
    print(f"  Minimum reward achieved: {min(rewards):.1f}")

# Plot the results
plot_learning_curve(episode_rewards, window=10)

---
# Part 4: Theory-Practice Bridge

Now let's connect every piece of theory to the implementation. This section will help you understand exactly where each equation lives in the code.

## 4.1 Mapping Equations to Code

### 1. Policy Network Output

**Theory**: The policy $\pi_\theta(a|s)$ is a probability distribution over actions given state.

**Code**:
```python
def pi(self, x, softmax_dim=0):
    features = self.forward_features(x)  # Extract features
    logits = self.fc_pi(features)        # Raw action scores
    probs = F.softmax(logits, dim=softmax_dim)  # Convert to probabilities
    return probs  # This is π_θ(a|s)
```

**Connection**: The softmax ensures the output is a valid probability distribution ($\sum_a \pi_\theta(a|s) = 1$).

---

### 2. Value Function

**Theory**: $V^\pi(s)$ estimates the expected return from state $s$.

**Code**:
```python
def v(self, x):
    features = self.forward_features(x)
    value = self.fc_v(features)  # This is V_θ(s)
    return value
```

**Connection**: Single output neuron produces a scalar value estimate.

---

### 3. TD Error Computation

**Theory**: $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$

**Code**:
```python
# In compute_gae function:
td_errors = rewards + gamma * next_values * dones - values
#           ^^^^^^    ^^^^^   ^^^^^^^^^^^   ^^^^^   ^^^^^^
#             r_t   +   γ   *   V(s_t+1)  * mask  - V(s_t)
```

**Connection**: 
- `dones` is a mask (1 if not done, 0 if done) to handle terminal states
- At terminal states: $\delta_T = r_T - V(s_T)$ (since $V(s_{\text{terminal}}) = 0$)

---

### 4. GAE Computation

**Theory**: $A_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$

**Recursive form**: $A_t = \delta_t + \gamma\lambda A_{t+1}$

**Code**:
```python
# In compute_gae function:
advantage = 0.0
for t in reversed(range(T)):  # Go backwards through time
    # A_t = δ_t + γλ * done_mask * A_{t+1}
    advantage = td_errors[t] + gamma * lmbda * dones[t] * advantage
    #           ^^^^^^^^^^^^   ^^^^^   ^^^^^   ^^^^^^^^   ^^^^^^^^^^
    #               δ_t     +    γ  *    λ   *   mask   *   A_{t+1}
    advantages[t] = advantage
```

**Connection**: 
- We iterate backwards because $A_t$ depends on $A_{t+1}$
- The `dones` mask ensures we don't carry advantages across episode boundaries

---

### 5. Advantage Normalization

**Theory**: While not in the original paper, normalizing advantages stabilizes training.

**Code**:
```python
# In train_ppo function:
advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
#             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#             Standardize to mean=0, std=1
```

**Why**: Makes the scale of advantages consistent across different environments and stages of training.

---

### 6. Probability Ratio

**Theory**: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$

**Code**:
```python
# In train_ppo function:
ratio = torch.exp(torch.log(pi_a) - torch.log(old_prob_a_batch))
#       ^^^^^^^^^  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#       exp(       log(π_θ(a_t|s_t)) - log(π_θ_old(a_t|s_t))      )
#
# Mathematical identity: a/b = exp(log(a) - log(b))
```

**Why log space?**: Numerical stability! Direct division can cause issues when probabilities are very small.

---

### 7. Clipped Surrogate Objective

**Theory**: $L^{\text{CLIP}} = \mathbb{E}_t[\min(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t)]$

**Code**:
```python
# In train_ppo function:
surr1 = ratio * advantages_batch
#       ^^^^^   ^^^^^^^^^^^^^^^^
#       r_t  *       A_t

surr2 = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantages_batch
#       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#       clip(r_t, 1-ε, 1+ε) * A_t

policy_loss = -torch.min(surr1, surr2).mean()
#             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#             -E[min(surr1, surr2)]  (negative because we maximize)
```

**Connection**:
- `surr1`: Unclipped objective
- `surr2`: Clipped objective  
- `torch.min`: Takes the pessimistic bound
- Negative sign: PyTorch minimizes loss, but we want to maximize objective

---

### 8. Value Function Loss

**Theory**: $L^V = \mathbb{E}_t[(V_\theta(s_t) - \hat{V}_t)^2]$

**Code**:
```python
# In train_ppo function:
value_loss = F.smooth_l1_loss(values_batch, returns_batch)
#            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#            SmoothL1(V_θ(s_t), V̂_t)
```

**Note**: We use Smooth L1 (Huber loss) instead of MSE for better robustness to outliers.

---

### 9. Entropy Bonus

**Theory**: $H(\pi) = -\sum_a \pi_\theta(a|s) \log \pi_\theta(a|s)$

**Code**:
```python
# In train_ppo function:
entropy = -(pi * torch.log(pi + 1e-8)).sum(dim=1).mean()
#          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#          -Σ π(a|s) * log(π(a|s))
```

**Why**: 
- Encourages exploration by preventing premature convergence to deterministic policy
- `1e-8` prevents log(0) which would be -inf

---

### 10. Total Loss

**Theory**: $L_{\text{total}} = -L^{\text{CLIP}} + c_1 L^V - c_2 H$

**Code**:
```python
# In train_ppo function:
total_loss = policy_loss + c1 * value_loss - c2 * entropy
#            ^^^^^^^^^^^   ^^^^^^^^^^^^^^^^   ^^^^^^^^^^^^^
#            -L^CLIP    +  c₁ * L^V       -  c₂ * H
```

**Connection**: All three components balanced by coefficients $c_1$ and $c_2$.

## 4.2 Data Flow Diagram

Let's trace how data flows through the algorithm:

```
1. TRAJECTORY COLLECTION
   State s_t ──→ Policy π_θ ──→ Action a_t ──→ Environment ──→ (s_{t+1}, r_t)
                      ↓
                  Store: (s_t, a_t, r_t, s_{t+1}, π_θ(a_t|s_t), done)

2. VALUE ESTIMATION
   States ──→ Value Network V_θ ──→ V(s_t), V(s_{t+1})

3. ADVANTAGE COMPUTATION
   (r_t, V(s_t), V(s_{t+1})) ──→ TD Errors δ_t ──→ GAE ──→ Advantages A_t

4. POLICY UPDATE
   (s_t, a_t, A_t, π_θ_old(a_t|s_t)) ──→ Clipped Objective ──→ Policy Gradient
                                                ↓
                                         Update θ

5. VALUE UPDATE  
   (V_θ(s_t), V̂_t) ──→ Value Loss ──→ Value Gradient ──→ Update θ
```

## 4.3 Hyperparameter Impact

Let's understand how each hyperparameter affects the algorithm:

| Hyperparameter | Symbol | Effect if Too Small | Effect if Too Large |
|----------------|--------|---------------------|---------------------|
| Clipping | $\epsilon$ | Very conservative updates, slow learning | Allows large policy changes, unstable |
| GAE Lambda | $\lambda$ | High bias, low variance | Low bias, high variance |
| Discount | $\gamma$ | Myopic (short-sighted) | Considers far future, slower |
| Learning Rate | $\alpha$ | Very slow learning | Unstable, divergence |
| Epochs | $K$ | Underutilizes data | Overfitting, overly conservative |
| Batch Size | - | Noisy gradients | Slow updates, less exploration |
| Value Coef | $c_1$ | Poor value estimates | Dominates policy learning |
| Entropy Coef | $c_2$ | Premature convergence | Too random, never converges |

## 4.4 Common Implementation Pitfalls

### ❌ Pitfall 1: Forgetting Done Masks
```python
# WRONG: Doesn't handle terminal states
td_errors = rewards + gamma * next_values - values

# CORRECT: Uses done mask
td_errors = rewards + gamma * next_values * dones - values
```

### ❌ Pitfall 2: Not Normalizing Advantages
Without normalization, advantages can have wildly different scales, making training unstable.

### ❌ Pitfall 3: Using Stale Probabilities
```python
# WRONG: Computing ratio with current policy twice
ratio = pi_a / pi_a  # Always equals 1!

# CORRECT: Using stored old probabilities
ratio = pi_a / old_prob_a
```

### ❌ Pitfall 4: Incorrect GAE Computation
```python
# WRONG: Forward iteration
for t in range(T):
    advantage = td_errors[t] + gamma * lmbda * advantage  # Uses future!

# CORRECT: Backward iteration
for t in reversed(range(T)):
    advantage = td_errors[t] + gamma * lmbda * advantage
```

---
# Part 5: Experiments and Analysis

## 5.1 Visualizing the Policy

Let's see what our trained policy has learned!

In [ ]:
def test_policy(model, n_episodes=5, render=False):
    """
    Test the trained policy.
    
    Args:
        model: Trained PPO model
        n_episodes: Number of episodes to test
        render: Whether to render the environment
    
    Returns:
        List of episode rewards
    """
    env = gym.make(env_name, render_mode='human' if render else None)
    test_rewards = []
    
    for episode in range(n_episodes):
        state, _ = env.reset()
        episode_reward = 0.0
        done = False
        
        while not done:
            # Select action greedily (no exploration)
            with torch.no_grad():
                state_tensor = torch.from_numpy(state).float()
                probs = model.pi(state_tensor)
                action = torch.argmax(probs).item()  # Greedy action
            
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
        
        test_rewards.append(episode_reward)
        print(f"Test Episode {episode + 1}: Reward = {episode_reward:.1f}")
    
    env.close()
    
    print(f"\nAverage Test Reward: {np.mean(test_rewards):.1f} ± {np.std(test_rewards):.1f}")
    return test_rewards

# Test the trained policy
print("Testing trained policy...\n")
test_rewards = test_policy(trained_model, n_episodes=10)

## 5.2 Understanding Clipping

Let's visualize how the clipping mechanism works.

In [ ]:
def visualize_clipping(eps_clip=0.2):
    """
    Visualize PPO's clipping mechanism.
    """
    # Create a range of probability ratios
    ratios = np.linspace(0.5, 2.0, 200)
    
    # Positive and negative advantages
    A_positive = 1.0
    A_negative = -1.0
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Case 1: Positive Advantage (good action)
    surr1_pos = ratios * A_positive
    surr2_pos = np.clip(ratios, 1 - eps_clip, 1 + eps_clip) * A_positive
    clipped_obj_pos = np.minimum(surr1_pos, surr2_pos)
    
    ax1.plot(ratios, surr1_pos, 'b-', linewidth=2, label='Unclipped: $r_t A_t$')
    ax1.plot(ratios, surr2_pos, 'r--', linewidth=2, label='Clipped: $clip(r_t) A_t$')
    ax1.plot(ratios, clipped_obj_pos, 'g-', linewidth=3, label='Final (min): $L^{CLIP}$')
    ax1.axvline(x=1-eps_clip, color='gray', linestyle=':', alpha=0.5)
    ax1.axvline(x=1+eps_clip, color='gray', linestyle=':', alpha=0.5)
    ax1.axvline(x=1, color='black', linestyle='-', alpha=0.3)
    ax1.fill_between([1-eps_clip, 1+eps_clip], -1, 2, alpha=0.1, color='green', 
                      label='Trust region')
    ax1.set_xlabel('Probability Ratio $r_t(\\theta)$', fontsize=12)
    ax1.set_ylabel('Objective Value', fontsize=12)
    ax1.set_title(f'PPO Clipping: Positive Advantage ($A_t > 0$)', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([-0.5, 2.5])
    
    # Case 2: Negative Advantage (bad action)
    surr1_neg = ratios * A_negative
    surr2_neg = np.clip(ratios, 1 - eps_clip, 1 + eps_clip) * A_negative
    clipped_obj_neg = np.minimum(surr1_neg, surr2_neg)
    
    ax2.plot(ratios, surr1_neg, 'b-', linewidth=2, label='Unclipped: $r_t A_t$')
    ax2.plot(ratios, surr2_neg, 'r--', linewidth=2, label='Clipped: $clip(r_t) A_t$')
    ax2.plot(ratios, clipped_obj_neg, 'g-', linewidth=3, label='Final (min): $L^{CLIP}$')
    ax2.axvline(x=1-eps_clip, color='gray', linestyle=':', alpha=0.5)
    ax2.axvline(x=1+eps_clip, color='gray', linestyle=':', alpha=0.5)
    ax2.axvline(x=1, color='black', linestyle='-', alpha=0.3)
    ax2.fill_between([1-eps_clip, 1+eps_clip], -2.5, 0.5, alpha=0.1, color='green',
                      label='Trust region')
    ax2.set_xlabel('Probability Ratio $r_t(\\theta)$', fontsize=12)
    ax2.set_ylabel('Objective Value', fontsize=12)
    ax2.set_title(f'PPO Clipping: Negative Advantage ($A_t < 0$)', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([-2.5, 0.5])
    
    plt.tight_layout()
    plt.show()
    
    print("\nKey Observations:")
    print(f"1. Trust region: [{1-eps_clip:.2f}, {1+eps_clip:.2f}]")
    print("2. Left plot (A > 0): Clipping prevents excessive probability increases")
    print("3. Right plot (A < 0): Clipping prevents excessive probability decreases")
    print("4. Green shaded area: Region where policy gradient is normal")
    print("5. Outside trust region: Gradient becomes zero (flat line)")

visualize_clipping(eps_clip=0.2)

## 5.3 Exercises

Now it's your turn to experiment! Try these exercises to deepen your understanding:

### Exercise 1: Hyperparameter Sensitivity
Modify the hyperparameters and observe the effects:
- What happens if you set `eps_clip = 0.05` (very conservative)?
- What happens if you set `eps_clip = 0.5` (very permissive)?
- Try different values of `lmbda` (0.9, 0.95, 0.99)
- Experiment with the number of epochs `K_epochs` (3, 5, 10, 20)

### Exercise 2: Remove Components
See what happens when you remove key components:
- Remove advantage normalization (comment out the normalization line)
- Set entropy coefficient to 0 (no exploration bonus)
- Remove clipping (just use the unclipped objective)
- Use Monte Carlo returns instead of GAE (set λ=1)

### Exercise 3: Architecture Changes
- Try a deeper network (add more hidden layers)
- Use separate networks for policy and value
- Change the hidden dimension (128, 256, 512)

### Exercise 4: Different Environment
- Try PPO on `MountainCar-v0` (harder problem!)
- Try PPO on `LunarLander-v2` (discrete actions)
- Adjust hyperparameters for each environment

### Exercise 5: Analysis
- Plot how the probability ratio changes during training
- Visualize the value function predictions over time
- Track how often clipping activates during training
- Compare training curves with and without mini-batching

In [ ]:
# Your experimentation code here!
# Try the exercises above and see what you learn.

# Example: Training with different epsilon
# eps_clip_test = 0.05  # Try this!
# test_rewards, test_model = train_agent(n_episodes=200, ...)


---
# Conclusion

## What You've Learned

Congratulations! 🎉 You've completed a deep dive into PPO. You now understand:

1. **The Motivation**: Why PPO was developed and what problems it solves
2. **The Mathematics**: 
   - How the clipped surrogate objective prevents destructive updates
   - How GAE balances bias and variance in advantage estimation
   - How all the components work together
3. **The Implementation**: 
   - How to build PPO from scratch in PyTorch
   - How mini-batch training improves sample efficiency
   - Important implementation details (normalization, masking, etc.)
4. **The Connection**: 
   - How every line of code implements specific equations
   - How to debug PPO implementations
   - Common pitfalls and how to avoid them

## Key Takeaways

1. **PPO is Simple**: No complicated constrained optimization like TRPO
2. **Clipping is Clever**: A simple min and clip operation creates an effective trust region
3. **Details Matter**: Advantage normalization, done masks, and other "small" details are crucial
4. **Balance is Key**: The $\epsilon$, $\lambda$, and $K$ hyperparameters balance exploration, stability, and sample efficiency

## Next Steps

Now that you understand PPO deeply, you can:

1. **Extend PPO**: 
   - Add continuous action spaces (use Gaussian policies)
   - Implement PPO with recurrent networks (for partial observability)
   - Try PPO with intrinsic motivation

2. **Study Related Algorithms**:
   - Compare PPO to TRPO (understand the tradeoffs)
   - Learn about SAC (soft actor-critic for continuous control)
   - Study off-policy methods (DQN, TD3)

3. **Apply to Real Problems**:
   - Robotics control
   - Game playing
   - Resource allocation

4. **Read Research Papers**:
   - You're now equipped to understand RL papers!
   - Start with the original PPO paper
   - Explore recent improvements and applications

## Final Thoughts

PPO represents a beautiful balance between theory and practice in reinforcement learning. Its elegant solution to the policy optimization problem - using simple clipping instead of complex constraints - demonstrates that sometimes the best solutions are the simplest ones.

The journey from understanding the motivation, through the mathematics, to a working implementation has given you more than just knowledge of PPO. You've learned how to:
- Translate mathematical theory into working code
- Debug algorithms by connecting theory and practice
- Design experiments to validate understanding
- Think critically about algorithmic design choices

These skills will serve you well throughout your RL journey and beyond.

**Keep learning, keep experimenting, and most importantly, keep having fun with reinforcement learning!** 🚀

---

## References

1. Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal Policy Optimization Algorithms. arXiv preprint arXiv:1707.06347.

2. Schulman, J., Levine, S., Abbeel, P., Jordan, M., & Moritz, P. (2015). Trust Region Policy Optimization. ICML.

3. Schulman, J., Moritz, P., Levine, S., Jordan, M., & Abbeel, P. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. ICLR.

4. OpenAI Spinning Up in Deep RL: https://spinningup.openai.com/

5. Stable-Baselines3 Documentation: https://stable-baselines3.readthedocs.io/

---

*This notebook was created with ❤️ for your study group. We hope it helps you master PPO!*